# Neural Networks


## Introduction

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/16-Neural-Networks.ipynb)

In [ ]:
# Setup for Colab and for a local clone of the Materials repo: make the course's
# helper modules importable, downloading any that are not already available.
import importlib.util, os, sys, urllib.request

_BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main"
for _path in ["scripts/slideUtilities.py"]:
    _name = os.path.splitext(os.path.basename(_path))[0]
    if importlib.util.find_spec(_name) is not None:
        continue
    for _root in (".", ".."):                      # a clone: notebook in jupyter_notebooks/
        if os.path.exists(os.path.join(_root, _path)):
            sys.path.insert(0, os.path.abspath(os.path.dirname(os.path.join(_root, _path))))
            break
    else:                                          # Colab: fetch the file
        os.makedirs("modules", exist_ok=True)
        urllib.request.urlretrieve(f"{_BASE}/{_path}", os.path.join("modules", os.path.basename(_path)))
        if os.path.abspath("modules") not in sys.path:
            sys.path.insert(0, os.path.abspath("modules"))


In [ ]:
#| code-fold: true
#| code-summary: "Imports and setup"
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning
from slideUtilities import OKABE_ITO

warnings.filterwarnings('ignore')
%matplotlib inline

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/NN-figs/IntroModels.svg)

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/NN-figs/IntroModels2a.svg)


One lecture, two halves: **how a network learns** (neuron, layers, loss, gradient descent, backpropagation, mini-batches), then **how to train one** with scikit-learn and read what the training curves say.

The slides are deliberately light: a headline, a figure, an equation or a code output. The explanation is in the recording. Backpropagation worked by hand through an autograd class, the California-housing regression, grid search and the limits of scikit-learn's MLP are in [Appendix 16A](16A-Appendix-NN-Walkthroughs.qmd); the full `Value` framework is [Module M12](M12-NN-Backprop.qmd); learning-rate schedules, `warm_start`, `partial_fit` and the same network in PyTorch are [Module M5](M5-NN-Scikit-Learn.qmd). Nothing in those is on the quiz.


## Lecture Organization

By the end of this lecture you can:

* explain how a **neuron** extends logistic regression, and why a network needs a non-linear **activation**;
* count the **parameters** of an MLP from its layer sizes;
* state the **training loop** — loss, gradient, update — and predict what the **learning rate** does;
* say what **backpropagation** computes and why it costs about one forward pass;
* train an `MLPClassifier`, **read its loss curve**, and tell a learning-rate problem from **overfitting**.


# From Regression to Networks

## From Logistic Regression to a Neuron

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/NN-figs/neuron_model.jpeg)

From [cs231n](https://cs231n.github.io/neural-networks-1/)

Logistic regression:

$$
P(y = 1 \mid \mathbf{x}) = \sigma(\mathbf{x}^T\boldsymbol\beta)
$$

A neuron:

$$
\text{out} = f\Big(\sum_{i} w_i x_i + b\Big)
$$

With $f = \sigma$, a neuron **is** logistic regression: weights $w_i$, bias $b$, squash.


Linear regression computes $\hat y = \mathbf{x}^T\boldsymbol\beta$ and can only model linear relationships. Logistic regression wraps that in a sigmoid. A neuron is the same thing with the activation $f$ left open: $f$ may be the sigmoid, $\tanh$, or ReLU. A network is many neurons in parallel (a **layer**) and layers in sequence.


## Activation Functions

In [ ]:
#| echo: false
#| fig-align: center
xs = np.linspace(-5, 5, 400)
acts = [('sigmoid', 1 / (1 + np.exp(-xs))), ('tanh', np.tanh(xs)), ('ReLU', np.maximum(0, xs))]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, (name, ys), c in zip(axes, acts, [OKABE_ITO[0], OKABE_ITO[2], OKABE_ITO[5]]):
    ax.plot(xs, ys, color=c, lw=2.5)
    ax.axhline(0, color='0.7', lw=0.8)
    ax.axvline(0, color='0.7', lw=0.8)
    ax.set_title(name)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Why an activation at all?**

Without one, every layer is an affine map, and a composition of affine maps is **one** affine map: a hundred layers would still be linear regression. ReLU, $\max(0, x)$, is the default today.


## The Multi-Layer Perceptron

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/NN-figs/neural_net2.jpeg)

From [cs231n](https://cs231n.github.io/convolutional-networks/)

$$
\begin{aligned}
\mathbf{h}_1 &= f(\boldsymbol\beta_0 + \boldsymbol\Omega_0 \mathbf{x}) \\
\mathbf{h}_2 &= f(\boldsymbol\beta_1 + \boldsymbol\Omega_1 \mathbf{h}_1) \\
\hat{\mathbf{y}} &= \boldsymbol\beta_2 + \boldsymbol\Omega_2 \mathbf{h}_2
\end{aligned}
$$

$\boldsymbol\Omega_k$: weight matrices. $\boldsymbol\beta_k$: bias vectors (UDL's notation; not regression coefficients).

The whole network is **one function** $\hat{\mathbf y} = g(\mathbf{x};\theta)$ of its parameters $\theta = \{\boldsymbol\Omega_k, \boldsymbol\beta_k\}$.


Every neuron in a layer takes every output of the previous layer as input, so this is also called a **fully connected** or **dense** network. Hidden layers learn intermediate representations; the output layer has no activation, because the loss (and, for classification, a softmax) sits on top of it.


## Counting Parameters

A layer with $D_{\text{in}}$ inputs and $D_{\text{out}}$ neurons has $D_{\text{in}} D_{\text{out}}$ weights and $D_{\text{out}}$ biases.

The 8×8 digits (64 pixels, 10 classes) with one hidden layer of 64, `hidden_layer_sizes=(64,)`:

| layer | weights | biases | parameters |
|---|---|---|---|
| $64 \to 64$ | $64 \times 64 = 4{,}096$ | $64$ | $4{,}160$ |
| $64 \to 10$ | $64 \times 10 = 640$ | $10$ | $650$ |
| **total** | | | **$4{,}810$** |


`(512, 512)` on the same data: $64{\cdot}512 + 512 + 512{\cdot}512 + 512 + 512{\cdot}10 + 10 = 301{,}066$. Keep that number for later.


# Training

## The Loss Function

Training means choosing $\theta$ to minimize a **loss** $L(\theta)$ over the training set.

**Regression** — mean squared error:

$$
L = \frac{1}{N}\sum_{i=1}^N (\hat y_i - y_i)^2
$$

**Classification** — cross-entropy, with a softmax on the output so the $\hat y_{ic}$ are probabilities:

$$
L = -\frac{1}{N}\sum_{i=1}^N \sum_{c=1}^C y_{ic} \log \hat y_{ic}
$$


Training is a loop: compute the **loss**, compute its **gradient** with respect to every parameter, **update** the parameters a little, repeat.


## Gradient Descent

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/L23-convex_cost_function.jpeg)

Left, convex (linear regression). Right, non-convex (a neural network): no closed form, so we *walk*.

Start from random $\mathbf{w}^{(0)}$ and repeat

$$
\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \eta\, \nabla_\mathbf{w} L(\mathbf{w}^{(t)})
$$

* $\nabla_\mathbf{w} L$: the vector of partials $\partial L/\partial w_j$, pointing **uphill**; the minus sign walks downhill.
* $\eta$: the **learning rate**, the step size.
* Stop when the loss (on held-out data) stops improving, or the budget runs out.


The foggy-mountain picture: look around, find the direction that slopes down most steeply, take a few steps, repeat until the ground is level. Linear regression had a closed form because its loss is a convex quadratic; a network's loss is a composition of non-linear layers, so the only thing we can do cheaply is evaluate $L$ and $\nabla L$ anywhere and step. [Module M11](M11-NN-Gradient-Descent.qmd) does gradient descent on linear regression step by step, with animations.


## The Learning Rate

On $L(w) = 3w^2 - 4w + 5$ (minimum at $w = 2/3$) the gradient is $6w - 4$, so each step multiplies the distance to the minimum by $(1 - 6\eta)$. Eight steps from $w = -3$:

In [ ]:
#| echo: false
#| fig-align: center
def f(x):
    return 3*x**2 - 4*x + 5

def df(x):
    return 6*x - 4

def descend(eta, w=-3.0, steps=8):
    path = [w]
    for _ in range(steps):
        w = w - eta * df(w)
        path.append(w)
    return np.array(path)

runs = [(0.1, 'η = 0.1: factor 0.4, converges'),
        (1/3, 'η = 1/3: factor −1, bounces forever'),
        (0.35, 'η = 0.35: factor −1.1, diverges')]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
xs = np.linspace(-8, 9, 200)
for ax, (eta, title) in zip(axes, runs):
    path = descend(eta)
    ax.plot(xs, f(xs), color=OKABE_ITO[5])
    ax.plot(path, f(path), 'o-', color=OKABE_ITO[3], markersize=4)
    ax.plot(path[0], f(path[0]), 'ks', markersize=6)
    ax.set_title(title)
    ax.set_xlabel('Parameter w')
    ax.set_ylim([0, 200])
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Loss L(w)')
plt.tight_layout()
plt.show()

**Too small** is merely slow. Past $\eta = 1/3$ every step overshoots by more than it started with, and the loss **grows**.

# Backpropagation

## Where Do the Gradients Come From?

Gradient descent needs $\partial L/\partial\theta$ for **every** weight and bias, and $L$ is a deeply nested composition:

$$
L = \ell\big(\,\boldsymbol\beta_K + \boldsymbol\Omega_K\, f(\boldsymbol\beta_{K-1} + \boldsymbol\Omega_{K-1}\, f(\cdots f(\boldsymbol\beta_0 + \boldsymbol\Omega_0 \mathbf{x})\cdots))\,,\ y\big)
$$

* By hand, for a million parameters: hopeless.
* Numerically, wiggling each parameter: one or two forward passes **per parameter** per step.
* Wanted: *all* the partials for a small constant multiple of **one** forward pass.


That is **backpropagation**: the chain rule, organized on the **computation graph**.


## One Tiny Graph

$d = a \cdot b$, $\ e = d + c$, $\ L = e \cdot f$, with $a = 4$, $b = -3$, $c = 8$, $f = 2$.

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/L16-recap-graph-forward.svg)

**Forward pass**, left to right: $d = -12$, $e = -4$, $L = -8$. Every node stores its value; every `grad` is still $0$.


Each box is one number with a slot for its gradient; the small nodes are the operations. This is exactly the structure PyTorch builds when you write `a * b + c`. [Appendix 16A](16A-Appendix-NN-Walkthroughs.qmd) builds it in twenty lines of Python and fills in every gradient by hand.


## The Chain Rule, Node by Node

$$
\frac{\partial L}{\partial c} = \frac{\partial L}{\partial e}\cdot\frac{\partial e}{\partial c}
\qquad \text{(gradient from above)} \times \text{(local derivative)}, \ \text{both at the forward-pass values.}
$$

* `L.grad = 1`. Then $L = e \cdot f$: $\partial L/\partial e = f = 2$ and $\partial L/\partial f = e = -4$. A **product** hands each operand the *other's* value.
* $e = d + c$: local derivative $1$. A **sum** just *routes* the gradient: $\partial L/\partial d = \partial L/\partial c = 2$.
* $d = a \cdot b$: $\partial L/\partial b = a \cdot 2 = 8$, $\ \partial L/\partial a = b \cdot 2 = -6$.


![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/L16-recap-graph-backward.svg)

Check: raise $b$ by $1$ and $L = (4\cdot(-2) + 8)\cdot 2 = 0$, up by $8$.


## The Backprop Recipe

1. **Forward pass**, storing every node's value — the local derivatives are evaluated *there*.
2. Set the output gradient to $1$.
3. **Backward pass**, right to left: each node multiplies the gradient it receives by its **local derivative** and hands the product to its children. `+` routes; `*` swaps in the other operand; `ReLU` passes if its input was $> 0$, else sends $0$; the loss $\tfrac12(\hat y - y)^2$ sends $\hat y - y$.


Every step is **local**, so each node is visited once, as in the forward pass: all the partials for a small constant multiple of one forward pass, however many parameters. *That is the whole idea.*


A ReLU whose input is negative sends $0$ to everything upstream of it: for that example, its weights get no gradient at all (a *dead ReLU*). The worked neuron, the two-layer network with a loss on top, one gradient step and the dead-ReLU case are in [Appendix 16A](16A-Appendix-NN-Walkthroughs.qmd#backprop-on-a-neuron); the automatic `backward()`, gradient accumulation, and a complete training loop are in [Module M12](M12-NN-Backprop.qmd).


## Batches: Full, Stochastic, Mini

Backprop gives the gradient for **one** sample, $\nabla_\mathbf{w}\ell_i$. How many samples before each step?

| | samples per step | gradient | noise |
|---|---|---|---|
| **full batch** | all $N$: $\nabla L = \frac1N\sum_i \nabla\ell_i$ | exact | none |
| **stochastic** (SGD) | $1$ | very rough | lots |
| **mini-batch** | $B \approx 32$–$256$: $\frac1B\sum_{i\in\text{batch}}\nabla\ell_i$ | good enough | some |

Full batch costs one pass over the data per step. Mini-batch is what everyone uses: cheap steps, GPU-friendly, and its noise can shake a non-convex descent out of a poor basin (observed, not a theorem).


## Comparing Batch Sizes

Linear regression, $N = 1000$ points, $\eta = 0.05$ for every batch size. Vertical axis: training MSE above the least-squares optimum.

In [ ]:
#| echo: false
#| fig-align: center
rng = np.random.default_rng(0)
N = 1000
X_b = rng.normal(size=(N, 2))
y_b = X_b @ np.array([2.0, -1.0]) + 0.5 * rng.normal(size=N)
best_mse = np.mean((X_b @ np.linalg.lstsq(X_b, y_b, rcond=None)[0] - y_b) ** 2)

def excess_mse(w):
    return np.mean((X_b @ w - y_b) ** 2) - best_mse

def minibatch_gd(B, steps, eta=0.05, seed=0):
    batch_rng = np.random.default_rng(seed)
    w = np.zeros(2)
    history = [excess_mse(w)]
    for _ in range(steps):
        idx = batch_rng.choice(N, size=B, replace=False)
        w = w - eta * 2 * X_b[idx].T @ (X_b[idx] @ w - y_b[idx]) / B
        history.append(excess_mse(w))
    return np.array(history)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for B, label, colour, width in [(1, 'Stochastic (B = 1)', OKABE_ITO[3], 0.8),
                                (32, 'Mini-batch (B = 32)', OKABE_ITO[0], 1.8),
                                (N, f'Full batch (B = {N})', OKABE_ITO[5], 2.5)]:
    axes[0].plot(minibatch_gd(B, steps=100), color=colour, linewidth=width, label=label)
    budget = minibatch_gd(B, steps=5 * N // B)
    axes[1].plot(np.arange(len(budget)) * B / N, budget, color=colour, linewidth=width)
for ax in axes:
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
axes[0].set_ylim(1e-5, 10)
axes[0].set_title('Same number of steps')
axes[0].set_xlabel('Gradient step')
axes[0].set_ylabel('Training MSE above the best fit')
axes[0].legend(loc='upper right')
axes[1].set_title('Same computation: 5 passes over the data')
axes[1].set_xlabel('Passes over the data (epochs)')
plt.tight_layout()
plt.show()

**Per step** (left) the full batch wins. **Per unit of computation** (right) it has taken only 5 steps while mini-batch is already at its noise floor.

# Training in Practice

## Every Idea Is Now an Argument

scikit-learn's `MLPClassifier` and `MLPRegressor`:

| concept | argument |
|---|---|
| layers and neurons | `hidden_layer_sizes=(64,)` — input and output sizes come from the data |
| activation $f$ | `activation='relu'` |
| learning rate $\eta$ | `learning_rate_init=0.001` |
| mini-batch size $B$ | `batch_size=200` |
| optimizer | `solver='adam'` (default; adapts the step per parameter), `'sgd'`, `'lbfgs'` |
| epochs and stopping | `max_iter`, `early_stopping`, `n_iter_no_change` |
| L2 regularization | `alpha=1e-4` |

`max_iter` is a *ceiling* in epochs, not a target: `adam` and `sgd` stop earlier once the training loss has not improved by `tol` for `n_iter_no_change` epochs. Learning-rate schedules, `warm_start` and `partial_fit` are in [Module M5](M5-NN-Scikit-Learn.qmd).


## Demo: the 8×8 Digits

`load_digits`: 1,797 handwritten digits, 64 pixel features with values 0–16, 10 classes. Every fit below takes under a second; the in-class case study uses the same split.

In [ ]:
#| code-fold: false
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

SEED = 701
digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.3,
    stratify=digits.target, random_state=SEED)

scaler = StandardScaler().fit(X_train)          # fit on the training split only
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"train {X_train_s.shape}, test {X_test_s.shape}")

In [ ]:
#| echo: false
#| fig-align: center
fig, axes = plt.subplots(1, 10, figsize=(11, 1.4))
for ax, img, lab in zip(axes, X_train[:10], y_train[:10]):
    ax.imshow(img.reshape(8, 8), cmap='gray_r')
    ax.set_title(str(lab), fontsize=10)
    ax.axis('off')
plt.show()

## Fit, and Read the Loss Curve

In [ ]:
#| code-fold: false
mlp = MLPClassifier(hidden_layer_sizes=(64,), activation='relu', solver='adam',
                    learning_rate_init=0.001, max_iter=300, random_state=SEED)
mlp.fit(X_train_s, y_train)
print(f"stopped after {mlp.n_iter_} epochs of max_iter={mlp.max_iter}; "
      f"parameters: {sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)}")

In [ ]:
#| echo: false
#| fig-align: center
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(np.arange(1, mlp.n_iter_ + 1), mlp.loss_curve_, color=OKABE_ITO[5], lw=2)
ax.set_xlabel('epoch')
ax.set_ylabel('training loss (cross-entropy)')
ax.set_title('(64,) MLP on the 8×8 digits: training loss per epoch')
ax.grid(alpha=0.3)
plt.show()

One point per epoch: a fast drop, then a smooth flattening, until `adam` decides the loss has stopped improving. This is the **training** loss; it says nothing about digits the model has not seen.

## Score on Held-Out Data

In [ ]:
#| code-fold: false
print(f"training accuracy: {mlp.score(X_train_s, y_train):.3f}")
print(f"test accuracy:     {mlp.score(X_test_s, y_test):.3f}")

In [ ]:
#| echo: false
#| fig-align: center
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(5.2, 4.2))
ConfusionMatrixDisplay.from_estimator(mlp, X_test_s, y_test, ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion matrix, test split')
plt.tight_layout()
plt.show()

Training accuracy $1.0$ and test accuracy about $0.98$: the network fits its training set perfectly and generalizes well here, because 1,257 training digits are plenty for 4,810 parameters. Rows are true digits, columns predictions; the few off-diagonal cells are the confusable shapes.


## Reading the Learning Rate from the Loss Curve

Same `(64,)` network, same 60-epoch budget, three values of `learning_rate_init`:

In [ ]:
#| echo: false
#| fig-align: center
lr_settings = [(1e-5, 'too low'), (1e-3, 'just right'), (1.0, 'too high')]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, (lr, label), color in zip(axes, lr_settings, [OKABE_ITO[1], OKABE_ITO[2], OKABE_ITO[3]]):
    m = MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=lr,
                      max_iter=60, random_state=SEED).fit(X_train_s, y_train)
    ax.plot(m.loss_curve_, color=color, lw=2.5)
    ax.set_title(f"{label} (lr = {lr:g})\ntest acc {m.score(X_test_s, y_test):.3f}, {m.n_iter_} epochs")
    ax.set_xlabel('epoch')
    ax.grid(alpha=0.3)
    if label == 'too high':
        ax.text(8, 1.4, f"starts at {m.loss_curve_[0]:.0f}, bounces,\nstalls, stops early")
axes[0].set_ylabel('training loss (y-axis clipped)')
axes[0].set_ylim(0, 3.2)
plt.tight_layout()
plt.show()

* **Too low:** a nearly straight, slowly falling line — thousands of epochs from done.
* **Just right:** fast drop, smooth flattening.
* **Too high:** starts absurdly high, goes *up* on some epochs, stalls, stops "because it stopped improving".


## What the Training Curve Cannot Show

A `(512, 512)` network — the 301,066 parameters from earlier — with no L2 penalty, trained on **50** digits, one epoch at a time with `partial_fit`, scoring held-out digits after every epoch:

In [ ]:
#| code-fold: true
#| code-summary: "Show the training loop"
from sklearn.metrics import log_loss

classes = np.unique(digits.target)
idx_tiny, idx_val = train_test_split(np.arange(len(y_train)), train_size=50,
                                     stratify=y_train, random_state=SEED)
X_tiny, y_tiny = X_train_s[idx_tiny], y_train[idx_tiny]
X_val, y_val = X_train_s[idx_val], y_train[idx_val]

big = MLPClassifier(hidden_layer_sizes=(512, 512), alpha=0.0, random_state=SEED)
train_loss, val_loss = [], []
for epoch in range(200):
    big.partial_fit(X_tiny, y_tiny, classes=classes)          # exactly one epoch
    train_loss.append(log_loss(y_tiny, big.predict_proba(X_tiny), labels=classes))
    val_loss.append(log_loss(y_val, big.predict_proba(X_val), labels=classes))
best_epoch = int(np.argmin(val_loss))

In [ ]:
#| echo: false
#| fig-align: center
fig, ax = plt.subplots(figsize=(9.5, 3.6))
ax.plot(train_loss, color=OKABE_ITO[5], lw=2.5, label='training loss (the 50 digits it trains on)')
ax.plot(val_loss, color=OKABE_ITO[0], lw=2.5, label=f'validation loss ({len(y_val)} digits it has never seen)')
ax.axvline(best_epoch, color='gray', ls='--', label=f'validation minimum (epoch {best_epoch})')
ax.annotate('training loss → 0:\nthe 50 digits are memorized', xy=(60, 0.0), xytext=(70, 0.9),
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate('validation loss stops improving\nand drifts back up', xy=(120, val_loss[120]),
            xytext=(100, 1.4), arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_ylim(0, None)
ax.set_xlabel('epoch')
ax.set_ylabel('cross-entropy loss')
ax.set_title('(512, 512) network, 50 training digits, alpha = 0')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.show()

## Early Stopping

* **Overfitting is only visible on data the optimizer never saw.**
* `early_stopping=True`: hold out `validation_fraction=0.1`, score every epoch, stop after `n_iter_no_change` epochs without improvement, keep the best weights.


**Diagnose** on held-out data, then **repair**:

| repair | scikit-learn | what it changes |
|---|---|---|
| L2 regularization | `alpha=0.01` | the objective |
| early stopping | `early_stopping=True` | the training loop |
| a smaller network | `hidden_layer_sizes=(64,)` | the capacity |
| more data | more rows | the constraints per weight |


## Why Feature Scaling Matters

The same `(64,)` network, same 60-epoch budget, on the Wisconsin breast-cancer data (30 measurements; `worst area` runs to 3,400 while `smoothness error` stays below 0.03):

In [ ]:
#| code-fold: true
#| code-summary: "Show the two fits"
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    cancer.data, cancer.target, test_size=0.3, stratify=cancer.target, random_state=SEED)
scaler_c = StandardScaler().fit(Xc_train)

fits = {}
for name, (A_tr, A_te) in {'standardized': (scaler_c.transform(Xc_train), scaler_c.transform(Xc_test)),
                           'raw': (Xc_train, Xc_test)}.items():
    m = MLPClassifier(hidden_layer_sizes=(64,), max_iter=60, random_state=SEED).fit(A_tr, yc_train)
    fits[name] = m
    print(f"{name:12s}: first-epoch loss {m.loss_curve_[0]:6.2f}   "
          f"final loss {m.loss_curve_[-1]:.3f}   test accuracy {m.score(A_te, yc_test):.3f}")

In [ ]:
#| echo: false
#| fig-align: center
fig, ax = plt.subplots(figsize=(9, 3.2))
for (name, m), c in zip(fits.items(), [OKABE_ITO[5], OKABE_ITO[3]]):
    ax.plot(np.arange(1, m.n_iter_ + 1), m.loss_curve_, color=c, lw=2, label=name)
ax.set_yscale('log')
ax.set_xlabel('epoch')
ax.set_ylabel('training loss (log scale)')
ax.set_title('Breast-cancer data, (64,) MLP: standardized vs raw features')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.show()

Gradient descent takes one step size for every direction. When one feature spans thousands and another hundredths, the wide columns dominate every weighted sum and every gradient, the loss surface is a long narrow valley, and no single learning rate suits both. Standardizing (zero mean, unit variance per feature, **fitted on the training split only**) fixes that; on the digits it barely mattered because every pixel was already on the same 0–16 scale. The California-housing regression in [Appendix 16A](16A-Appendix-NN-Walkthroughs.qmd#why-scaling-matters-california-housing) shows the same effect on a regression target.


## Best Practices

* **Scale the inputs**, inside a `Pipeline([('scaler', StandardScaler()), ('mlp', MLPClassifier(...))])` so cross-validation refits the scaler per fold.
* **Start simple**: one hidden layer, sizes between the input and output dimensions; `solver='adam'`; add depth only when the data can pay for it.
* **Read the loss curve before the accuracy**: still falling → raise `max_iter`; bouncing → lower `learning_rate_init`; huge from epoch 1 → check scaling. A `ConvergenceWarning` is about the optimizer, not the model.
* **Hold data out**: a validation split or cross-validation is the only way to see overfitting; tune on folds, never on the test set.
* **Know the tool's limits**: scikit-learn's MLP has no dropout, no GPU, no convolutional layers. For images, sequences and large data: PyTorch ([Modules M4](M4-CNNs.qmd), [M3](M3-RNN.qmd), DS542).


## Summary

* A **neuron** is a weighted sum followed by an activation; a sigmoid neuron *is* logistic regression. Without a non-linear activation, stacked layers collapse to one linear map.
* An MLP is one composed function of its parameters; a layer $D_{\text{in}} \to D_{\text{out}}$ has $D_{\text{in}} D_{\text{out}} + D_{\text{out}}$ of them.
* **Training** minimizes a loss (MSE or cross-entropy) by **gradient descent**, $\theta \leftarrow \theta - \eta\nabla_\theta L$; too large an $\eta$ overshoots and diverges.
* **Backpropagation** runs the chain rule backwards over the compute graph — `+` routes, `*` swaps, ReLU gates — and gets every partial for about one forward pass.
* **Mini-batches** trade an exact gradient for cheap, slightly noisy steps.
* In scikit-learn: scale the inputs, read the **training loss curve** for the learning rate, hold data out to see **overfitting**, and repair with `alpha`, early stopping, a smaller network or more data.

## To Dig Deeper

In the course notes:

* [Appendix 16A](16A-Appendix-NN-Walkthroughs.qmd): backprop by hand through a 20-line autograd class — a neuron, a network with a loss, one gradient step, a dead ReLU; the California-housing regressor; grid search; what scikit-learn's MLP cannot do
* [Module M11: Gradient Descent](M11-NN-Gradient-Descent.qmd): gradient descent on linear regression, step by step, with animations
* [Module M12: the full `Value` framework](M12-NN-Backprop.qmd): automatic `backward()`, gradient accumulation, layers, a training loop, and the same thing in PyTorch
* [Module M5: scikit-learn's MLP, advanced features](M5-NN-Scikit-Learn.qmd): learning-rate schedules, `warm_start`, `partial_fit`, and the same MLP in PyTorch
* [Module M4: CNNs](M4-CNNs.qmd) and [Module M3: RNNs](M3-RNN.qmd)

Elsewhere:

* [Understanding Deep Learning, Simon J.D. Prince, MIT Press, 2023](http://udlbook.com), Chapters 3–7
* DS542, Deep Learning for Data Science